# Part 2: Swaption Calibration

In [1]:
# pip install pysabr

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pylab as plt
from scipy.optimize import least_squares
import warnings
import pysabr
from pysabr import Hagan2002LognormalSABR

## Import Dataset

In [3]:
sheets = pd.ExcelFile('Swap and Swaption Markets_Amended.xlsx').sheet_names

In [4]:
df_swaption = pd.read_excel('Swap and Swaption Markets_Amended.xlsx', sheet_name=sheets[3], header = 2)
df_swaption

,Expiry,Tenor,-200bps,-150bps,-100bps,-50bps,-25bps,ATM,+25bps,+50bps,+100bps,+150bps,+200bps
0,1Y,1Y,91.570,62.030,44.130,31.224,26.182,22.50,20.96,21.40,24.34,27.488,30.297
1,1Y,2Y,83.270,61.240,46.570,35.807,31.712,28.72,27.12,26.84,28.51,31.025,33.523
2,1Y,3Y,73.920,56.870,44.770,35.745,32.317,29.78,28.29,27.80,28.77,30.725,32.833
3,1Y,5Y,55.190,44.640,36.510,30.242,27.851,26.07,24.98,24.56,25.12,26.536,28.165
4,1Y,10Y,41.180,35.040,30.207,26.619,25.351,24.47,23.98,23.82,24.25,25.204,26.355
5,5Y,1Y,67.800,49.090,38.400,31.485,29.060,27.26,26.04,25.32,24.94,25.320,25.980
6,5Y,2Y,57.880,46.410,39.033,33.653,31.531,29.83,28.56,27.65,26.71,26.540,26.760
7,5Y,3Y,53.430,44.440,38.180,33.437,31.536,29.98,28.76,27.82,26.67,26.200,26.150
8,5Y,5Y,41.990,36.524,32.326,29.005,27.677,26.60,25.73,25.02,24.06,23.570,23.400
9,5Y,10Y,34.417,30.948,28.148,25.954,25.136,24.51,23.99,23.56,22.91,22.490,22.250


Notes on Swaption Data on first inspection:
* This is lognormal Implied Vol for SOFR Swaptions
* Convection for bps strike is (Forward + Basis Point)

In [5]:
def convert_tenor_to_numeric(series):
    num = series.str.slice(stop = -1).astype(int)
    freq = series.str.slice(start = -1)

    conditions = [freq.str.upper() == 'M', freq.str.upper() == 'Y']
    choices = [num * 30 / 360, num.astype(float)]

    return pd.Series(np.select(conditions, choices, default=np.nan), name=series.name)

In [6]:
def convert_bps_to_numeric(column_series):

    column_names = column_series.str.slice(stop = -3)
    replaced_list = []

    for col in column_names:
        if col == '':
            replaced_list.append(0.0)
        else:
            replaced_list.append(float(col) / 10000)
    
    return replaced_list  

In [7]:
# Extract the expiry and tenor information from the column names and convert them to numeric values
# Imp vol is calculated in percentages, so we need to divide by 100 to get the decimal form
# bps is defined as 1/100 of a percentage point, so we need to divide by 10000 to get the decimal form

df_timings = pd.concat([convert_tenor_to_numeric(df_swaption['Expiry']), convert_tenor_to_numeric(df_swaption['Tenor'])], axis=1)
df_bps = df_swaption.drop(columns = ['Expiry', 'Tenor']) / 100  # e.g. 22.5 -> 0.225
df_bps.columns = convert_bps_to_numeric(df_bps.columns)

display(pd.concat([df_timings, df_bps], axis=1))

,Expiry,Tenor,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,1.0,2.0,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,1.0,3.0,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,1.0,5.0,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,1.0,10.0,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,5.0,1.0,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,5.0,2.0,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,5.0,3.0,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,5.0,5.0,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,5.0,10.0,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


We have to pull back the discount factors to calculate annuities. We will use the OIS backcalculated swap discount rates for now

$$
A_{n+1,N}(t) = \sum^{N}_{i=n+1} \Delta_{i-1} D_{i}(t)
$$

$$
\therefore A_{n+1,N}(0) = \sum^{N}_{i=n+1} \Delta_{i-1} D_{i}(0) = \sum^{N}_{i=n+1} \Delta_{i-1} D(0,T_i)
$$

In [8]:
ois_discount_factors = pd.read_csv('ois_sofr_discounted_factors.csv', index_col=0)
ois_discount_factors = ois_discount_factors.drop(columns = 'Market Rate')
ois_discount_factors = ois_discount_factors.loc[round(ois_discount_factors['Tenor_Year_Numeric']) == ois_discount_factors['Tenor_Year_Numeric']].reset_index(drop = True)
display(ois_discount_factors.head(), ois_discount_factors.tail(1))

,Tenor_Year_Numeric,Discount Factor
0,1.0,0.966224
1,2.0,0.935300
2,3.0,0.903758
3,4.0,0.871806
4,5.0,0.839750


,Tenor_Year_Numeric,Discount Factor
29,30.0,0.289438


In [9]:
# Calculate Annuities
# Discount Factor Table is annual frequency!

def calculate_annuities(df_timings, discount_factors):

    annuity =\
    (
        pd.Series(
                    np.zeros(len(df_timings)),
                    name = 'Annuities'
                )
    )

    for row in df_timings.itertuples():
        annuity[row.Index] = np.sum(discount_factors[int(row[1]):int(row[1]+row[2])])

    return annuity

In [10]:
df_annuities = calculate_annuities(df_timings, ois_discount_factors['Discount Factor'])
pd.concat([df_timings, df_annuities, df_bps], axis = 1)

,Expiry,Tenor,Annuities,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.935300,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,1.0,2.0,1.839058,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,1.0,3.0,2.710864,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,1.0,5.0,4.358245,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,1.0,10.0,7.930389,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,5.0,1.0,0.807631,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,5.0,2.0,1.583386,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,5.0,3.0,2.327821,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,5.0,5.0,3.725251,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,5.0,10.0,6.723663,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


The Forward Prices at time 0 for swaps can also be calculated
$$
S_{n,N}(t) = \frac{D_n(t) - D_N(t)}{A_{n+1,N}}
$$

$$
\therefore S_{n,N}(0) = \frac{D_n(0) - D_N(0)}{A_{n+1,N}} = \frac{D(0,T_n) - D(0,T_N)}{A_{n+1,N}}
$$

In [11]:
# Calculate Forward Rates
# Discount Factors is on annual basis

def calculate_forward_rates(df_timings, annuities, discount_factors):

    df_timings = df_timings.astype(int)

    forward_rates =\
    (
        pd
        .Series(np.full(len(df_annuities), np.nan),
                name = "Forward")
    )

    for row in df_timings.itertuples():
        forward_rates[row.Index] = (discount_factors.iloc[row[1]] - discount_factors.iloc[row[1]+row[2]]) / annuities[row.Index]

    return forward_rates

In [12]:
df_forward_rates = calculate_forward_rates(df_timings, df_annuities, ois_discount_factors['Discount Factor'])

In [13]:
pd.concat([df_timings, df_annuities, df_forward_rates, df_bps], axis = 1)

,Expiry,Tenor,Annuities,Forward,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.935300,0.033724,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,1.0,2.0,1.839058,0.034525,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,1.0,3.0,2.710864,0.035247,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,1.0,5.0,4.358245,0.036608,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,1.0,10.0,7.930389,0.039007,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,5.0,1.0,0.807631,0.039468,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,5.0,2.0,1.583386,0.039911,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,5.0,3.0,2.327821,0.040339,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,5.0,5.0,3.725251,0.041100,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,5.0,10.0,6.723663,0.042281,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


We can also output the actual prices of the swaptions using the Black Scholes Lognormal model, which is the same as the Black76 model for swaptions.

In [14]:
## Default Functions for Black Scholes Lognormal Model. 
## In the event of a Black76 model, set S = F and r = 0.0.
## If there is dividend, set r = r-q where q is the dividend yield.

def BlackScholesLognormalPayer(S, K, r, sigma, T):
    d1 = (np.log(S/K)+(r+sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)


def BlackScholesLognormalReceiver(S, K, r, sigma, T):
    d1 = (np.log(S/K)+(r+sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

In [15]:
def calculate_prices(df_bps, df_timings, df_forward_rates, df_annuities):
    F = df_forward_rates
    annuity = df_annuities
    r = 0.0          # Black76 discounting convention

    strike_shifts = df_bps.columns.astype(float).to_list()
    T_vec = df_timings['Expiry'].astype(float).values

    payer_strike_shifts = [shift for shift in strike_shifts if shift >= 0]
    receiver_strike_shifts = [shift for shift in strike_shifts if shift < 0]

    # print(payer_strike_shifts, receiver_strike_shifts)

    payer_prices = pd.DataFrame(index=df_bps.index, columns=strike_shifts, dtype=float).drop(columns = receiver_strike_shifts)
    receiver_prices = pd.DataFrame(index=df_bps.index, columns=strike_shifts, dtype=float).drop(columns = payer_strike_shifts)

    for shift in payer_strike_shifts:
        K = F + shift
        vols = df_bps[shift].astype(float).values

        # print(vols)

        payer_prices[shift] = annuity * BlackScholesLognormalPayer(F, K, r, vols, T_vec)


    for shift in receiver_strike_shifts:
        K = F + shift
        vols = df_bps[shift].astype(float).values

        receiver_prices[shift] = annuity * BlackScholesLognormalReceiver(F, K, r, vols, T_vec)


    return payer_prices, receiver_prices, pd.concat([df_timings, receiver_prices, payer_prices], axis=1)

In [16]:
# Attach expiry/tenor for readability
payer_swaption_prices, receiver_swaption_prices, df_swaption_prices = calculate_prices(df_bps, df_timings, df_forward_rates, df_annuities)
display(df_swaption_prices)

,Expiry,Tenor,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.001488,0.001299,0.001391,0.001748,0.002129,0.002825,0.001718,0.001128,0.000637,0.000434,0.000322
1,1.0,2.0,0.002504,0.002696,0.003306,0.004547,0.005624,0.007250,0.005031,0.003568,0.002081,0.001416,0.001053
2,1.0,3.0,0.002850,0.003509,0.004719,0.006936,0.008744,0.011310,0.008063,0.005789,0.003294,0.002142,0.001527
3,1.0,5.0,0.001971,0.003138,0.005178,0.009016,0.012162,0.016546,0.011519,0.007961,0.004033,0.002303,0.001457
4,1.0,10.0,0.001430,0.003186,0.006916,0.014780,0.021308,0.030123,0.021567,0.015273,0.007791,0.004235,0.002482
5,5.0,1.0,0.006146,0.005593,0.005707,0.006342,0.006892,0.007633,0.006564,0.005698,0.004468,0.003681,0.003141
6,5.0,2.0,0.009677,0.010296,0.011676,0.013675,0.014957,0.016510,0.014419,0.012632,0.009895,0.008016,0.006691
7,5.0,3.0,0.012778,0.014461,0.016950,0.020236,0.022267,0.024651,0.021610,0.018949,0.014732,0.011710,0.009549
8,5.0,5.0,0.013594,0.017272,0.021973,0.027953,0.031598,0.035802,0.031244,0.027201,0.020652,0.015856,0.012393
9,5.0,10.0,0.017247,0.024435,0.033681,0.045648,0.052999,0.061387,0.053835,0.047115,0.035946,0.027425,0.021045


There are two ways to calibrate the model:

1. Calibrate the model to the prices derived from the implied vol reporting models - i.e.: Output Black from implied vol to price, then use calibrate pricing model based on the outputted prices
2. Calibrate the model based on the implied vol reporting models directly

## Swaption Calibration

### A: Displaced-Diffusion Model 

This model can be built on top of the Black lognormal Model

$$
V_{n,N}(0) = P_{n+1, N}(0) \cdot Black\left(\frac{S_{n,N}(0)}{\beta}, K+\frac{1-\beta}{\beta}S_{n,N}(0), \sigma\beta, T \right)
$$

Given the above, this can be optimised. Objective is to optimise for $\beta$ and $\sigma$

In [17]:
(df_bps.head(), df_annuities.head(), df_forward_rates.head(), df_timings.head(), df_swaption_prices.head());

In [29]:
from scipy.optimize import least_squares
import warnings

def displaced_diffusion_payer(F, K, sigma_beta, T, beta):
    """
    Price a payer swaption using the displaced-diffusion model.
    
    Formula: V = Black(F/beta, K + (1-beta)/beta*F, sigma*beta, T)
    
    Parameters:
    -----------
    F : float
        Forward swap rate
    K : float
        Strike rate
    sigma_beta : float
        sigma * beta (volatility * displacement parameter)
    T : float
        Time to expiry (in years)
    beta : float
        Displacement parameter (0 < beta <= 1)
    
    Returns:
    --------
    float : Payer swaption price
    """
    if beta <= 0 or beta > 1:
        return np.nan
    
    # Adjusted forward and strike for displaced-diffusion
    F_adj = F / beta
    K_adj = K + (1 - beta) / beta * F
    sigma = sigma_beta / beta
    
    # Black-Scholes Lognormal pricing
    d1 = (np.log(F_adj / K_adj) + (sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    price = F_adj * norm.cdf(d1) - K_adj * norm.cdf(d2)
    return price


def displaced_diffusion_receiver(F, K, sigma_beta, T, beta):
    """
    Price a receiver swaption using the displaced-diffusion model.
    
    Parameters:
    -----------
    F : float
        Forward swap rate
    K : float
        Strike rate
    sigma_beta : float
        sigma * beta
    T : float
        Time to expiry (in years)
    beta : float
        Displacement parameter (0 < beta <= 1)
    
    Returns:
    --------
    float : Receiver swaption price
    """
    if beta <= 0 or beta > 1:
        return np.nan
    
    # Adjusted forward and strike
    F_adj = F / beta
    K_adj = K + (1 - beta) / beta * F
    sigma = sigma_beta / beta
    
    # Black-Scholes Lognormal pricing
    d1 = (np.log(F_adj / K_adj) + (sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    price = K_adj * norm.cdf(-d2) - F_adj * norm.cdf(-d1)
    return price


def calibrate_displaced_diffusion(
    market_prices,
    annuity,
    forward_rate,
    time_to_expiry,
    option_type='payer',
    beta_bounds=(0.01, 1.0),
    sigma_bounds=(0.001, 2.0),
    initial_guess=None,
    verbose=False
):
    """
    Calibrate the displaced-diffusion model for a single swaption.
    
    Calibrates both beta (displacement parameter) and sigma (volatility) 
    simultaneously using least-squares optimization.
    
    Parameters:
    -----------
    market_prices : array-like
        Market prices for different strikes (annuity-adjusted)
    annuity : float
        Present value of annuity (used for scaling)
    forward_rate : float
        Forward swap rate
    time_to_expiry : float
        Time to expiry in years
    option_type : str
        'payer' or 'receiver'
    beta_bounds : tuple
        (min_beta, max_beta) bounds for displacement parameter
    sigma_bounds : tuple
        (min_sigma, max_sigma) bounds for volatility
    initial_guess : tuple, optional
        Initial guess for (beta, sigma). Default: (0.7, 0.3)
    verbose : bool
        Print optimization details
    
    Returns:
    --------
    dict : Dictionary with keys:
        - 'beta': Calibrated displacement parameter
        - 'sigma': Calibrated volatility
        - 'residual': Sum of squared residuals
        - 'success': Whether optimization was successful
        - 'message': Optimization result message
    """
    
    if option_type.lower() not in ['payer', 'receiver']:
        raise ValueError("option_type must be 'payer' or 'receiver'")
    
    # Set pricing function based on option type
    price_func = displaced_diffusion_payer if option_type.lower() == 'payer' else displaced_diffusion_receiver
    
    # Initial guess for (beta, sigma)
    if initial_guess is None:
        initial_guess = [0.7, 0.3]
    
    # Objective function: residuals between market and model prices
    def objective(params):
        beta, sigma = params
        
        # Adjust prices by annuity
        K_strikes = forward_rate + np.array(market_prices.index)  # strikes
        
        residuals = []
        for strike, market_price in zip(K_strikes, market_prices.values):
            sigma_beta = sigma * beta
            model_price = price_func(forward_rate, strike, sigma_beta, time_to_expiry, beta)
            
            if np.isnan(model_price):
                return np.full_like(market_prices.values, 1e6)
            
            residuals.append((annuity * model_price - market_price) / annuity)
        
        return np.array(residuals)
    
    # Parameter bounds
    bounds = (
        [beta_bounds[0], sigma_bounds[0]],
        [beta_bounds[1], sigma_bounds[1]]
    )
    
    # Optimization
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = least_squares(
            objective,
            initial_guess,
            bounds=bounds,
            max_nfev=5000,
            ftol=1e-10,
            xtol=1e-10
        )
    
    calib_beta, calib_sigma = result.x
    residual_norm = np.sum(result.fun**2)
    
    if verbose:
        print(f"Calibration Results:")
        print(f"  Beta: {calib_beta:.6f}")
        print(f"  Sigma: {calib_sigma:.6f}")
        print(f"  Residual Norm: {residual_norm:.8e}")
        print(f"  Success: {result.success}")
        print(f"  Message: {result.message}")
    
    return {
        'beta': calib_beta,
        'sigma': calib_sigma,
        'residual': residual_norm,
        'success': result.success,
        'message': result.message,
        'nfev': result.nfev
    }


def calibrate_displaced_diffusion_batch(
    payer_prices,
    receiver_prices,
    df_annuities,
    df_forward_rates,
    df_timings,
    beta_bounds=(0.01, 1.0),
    sigma_bounds=(0.001, 2.0),
    initial_guess=None,
    verbose=False
):
    """
    Calibrate the displaced-diffusion model for all instruments (batch).
    
    Parameters:
    -----------
    payer_prices : DataFrame
        Payer swaption prices by strike (columns) and instrument (rows)
    receiver_prices : DataFrame
        Receiver swaption prices by strike (columns) and instrument (rows)
    df_annuities : Series
        Annuities for each instrument
    df_forward_rates : Series
        Forward rates for each instrument
    df_timings : DataFrame
        Time to expiry for each instrument (Expiry column)
    beta_bounds : tuple
        Bounds for beta parameter
    sigma_bounds : tuple
        Bounds for sigma parameter
    initial_guess : tuple, optional
        Initial guess for (beta, sigma)
    verbose : bool
        Print results for each instrument
    
    Returns:
    --------
    DataFrame : Calibration results with columns [beta, sigma, residual, success]
    """
    
    calibration_results = []
    
    for idx in payer_prices.index:
        expiry = df_timings.loc[idx, 'Expiry']
        forward = df_forward_rates.loc[idx]
        annuity = df_annuities.loc[idx]
        
        # Combine payer and receiver prices
        combined_prices = pd.concat([
            receiver_prices.loc[idx].dropna(),
            payer_prices.loc[idx].dropna()
        ])
        
        if len(combined_prices) == 0:
            continue
        
        # Determine option type based on first available price
        option_type = 'receiver' if receiver_prices.loc[idx].notna().any() else 'payer'
        
        # Calibrate
        result = calibrate_displaced_diffusion(
            combined_prices,
            annuity,
            forward,
            expiry,
            option_type=option_type,
            beta_bounds=beta_bounds,
            sigma_bounds=sigma_bounds,
            initial_guess=initial_guess,
            verbose=False
        )
        
        result['instrument'] = idx
        result['expiry'] = expiry
        result['forward'] = forward
        calibration_results.append(result)
        
        if verbose:
            print(f"\nInstrument {idx} (Expiry: {expiry:.4f}y):")
            print(f"  Beta: {result['beta']:.6f}, Sigma: {result['sigma']:.6f}")
            print(f"  Residual: {result['residual']:.8e}, Success: {result['success']}")
    
    return pd.DataFrame(calibration_results).set_index('instrument')


### Example: Calibrate Single Instrument

In [38]:
# Example 1: Calibrate a single swaption (first instrument, payer prices)
idx = 0
market_prices_single = payer_swaption_prices.iloc[idx].dropna()
market_prices_single_2 = receiver_swaption_prices.iloc[idx].dropna()

result_single = calibrate_displaced_diffusion(
    market_prices=market_prices_single,
    annuity=df_annuities.iloc[idx],
    forward_rate=df_forward_rates.iloc[idx],
    time_to_expiry=df_timings['Expiry'].iloc[idx],
    option_type='payer',
    initial_guess=[0.7, 0.3],
    verbose=True
)

result_single_2 = calibrate_displaced_diffusion(
    market_prices=market_prices_single_2,
    annuity=df_annuities.iloc[idx],
    forward_rate=df_forward_rates.iloc[idx],
    time_to_expiry=df_timings['Expiry'].iloc[idx],
    option_type='receiver',
    initial_guess=[0.7, 0.3],
    verbose=True
)

print(f"\nSummary:")
print(f"  Calibrated Beta: {result_single['beta']:.6f}")
print(f"  Calibrated Sigma: {result_single['sigma']:.6f}")
print(f"  Calibrated Beta (Receiver): {result_single_2['beta']:.6f}")
print(f"  Calibrated Sigma (Receiver): {result_single_2['sigma']:.6f}")

Calibration Results:
  Beta: 1.000000
  Sigma: 0.222564
  Residual Norm: 2.27050113e-07
  Success: True
  Message: `gtol` termination condition is satisfied.
Calibration Results:
  Beta: 0.010000
  Sigma: 0.003087
  Residual Norm: 4.21602585e-06
  Success: True
  Message: `gtol` termination condition is satisfied.

Summary:
  Calibrated Beta: 1.000000
  Calibrated Sigma: 0.222564
  Calibrated Beta (Receiver): 0.010000
  Calibrated Sigma (Receiver): 0.003087


### Example: Calibrate All Instruments (Batch)

In [36]:
# Example 2: Calibrate all instruments at once
calibration_results_batch = calibrate_displaced_diffusion_batch(
    payer_prices=payer_swaption_prices,
    receiver_prices=receiver_swaption_prices,
    df_annuities=df_annuities,
    df_forward_rates=df_forward_rates,
    df_timings=df_timings,
    beta_bounds=(0.01, 1.0),
    sigma_bounds=(0.001, 2.0),
    verbose=True
)

display(calibration_results_batch)


Instrument 0 (Expiry: 1.0000y):
  Beta: 0.022186, Sigma: 0.002705
  Residual: 7.16448598e-04, Success: True

Instrument 1 (Expiry: 1.0000y):
  Beta: 0.010219, Sigma: 0.001603
  Residual: 6.91247157e-04, Success: True

Instrument 2 (Expiry: 1.0000y):
  Beta: 0.010000, Sigma: 0.001614
  Residual: 6.88184995e-04, Success: True

Instrument 3 (Expiry: 1.0000y):
  Beta: 0.010454, Sigma: 0.001450
  Residual: 7.04214991e-04, Success: True

Instrument 4 (Expiry: 1.0000y):
  Beta: 0.012284, Sigma: 0.001585
  Residual: 7.00897867e-04, Success: True

Instrument 5 (Expiry: 5.0000y):
  Beta: 0.010000, Sigma: 0.001723
  Residual: 6.12959448e-04, Success: True

Instrument 6 (Expiry: 5.0000y):
  Beta: 0.010000, Sigma: 0.001852
  Residual: 5.79889654e-04, Success: True

Instrument 7 (Expiry: 5.0000y):
  Beta: 0.010000, Sigma: 0.001835
  Residual: 5.76777103e-04, Success: True

Instrument 8 (Expiry: 5.0000y):
  Beta: 0.010000, Sigma: 0.001539
  Residual: 5.65503973e-04, Success: True

Instrument 9 (Expi

,beta,sigma,residual,success,message,nfev,expiry,forward
instrument,,,,,,,,
0,0.022186,0.002705,0.000716,True,`gtol` termination condition is satisfied.,29,1.0,0.033724
1,0.010219,0.001603,0.000691,True,`gtol` termination condition is satisfied.,47,1.0,0.034525
2,0.010000,0.001614,0.000688,True,`gtol` termination condition is satisfied.,57,1.0,0.035247
3,0.010454,0.001450,0.000704,True,`gtol` termination condition is satisfied.,55,1.0,0.036608
4,0.012284,0.001585,0.000701,True,`gtol` termination condition is satisfied.,40,1.0,0.039007
5,0.010000,0.001723,0.000613,True,`gtol` termination condition is satisfied.,13,5.0,0.039468
6,0.010000,0.001852,0.000580,True,`gtol` termination condition is satisfied.,11,5.0,0.039911
7,0.010000,0.001835,0.000577,True,`gtol` termination condition is satisfied.,11,5.0,0.040339
8,0.010000,0.001539,0.000566,True,`gtol` termination condition is satisfied.,13,5.0,0.041100


### B: SABR Model

The implied Black volatility of the SABR model is given below, where $\beta = 0.75$ as a default setting

$$
    \begin{split}
      \sigma_{SABR}(F_0, K, \alpha, \beta, \rho, \nu)
      = \frac{\alpha}{(F_0K)^{(1-\beta)/2}\left\{ 1 + \frac{(1-\beta)^2}{24}\log^2\left(\frac{F_0}{K}\right) + \frac{(1-\beta)^4}{1920}\log^4\left(\frac{F_0}{K}\right) + \cdots\right\} }
      \times \frac{z}{x(z)} \times \left\{ 1 + \left[
           \frac{(1-\beta)^2}{24}
           \frac{\alpha^2}{(F_0K)^{1-\beta}}+\frac{1}{4}\frac{\rho\beta\nu\alpha}{(F_0K)^{(1-\beta)/2}}+\frac{2-3\rho^2}{24}\nu^2\right]
         T + \cdots \right.
    \end{split}
$$

where

$$
    \begin{split}
      z = \frac{\nu}{\alpha} (F_0K)^{(1-\beta)/2}
      \log\left(\frac{F_0}{K}\right),
    \end{split}
$$

and

$$
    % \begin{split}
      x(z) = \log \left[ \frac{\sqrt{1-2\rho z+z^2}+z -\rho}{1-\rho}
      \right].
    % \end{split}
$$

In [22]:
def SABR(F, K, T, alpha, beta, rho, nu):
    X = K
    # if K is at-the-money-forward
    if abs(F - K) < 1e-12:
        numer1 = (((1 - beta)**2)/24)*alpha*alpha/(F**(2 - 2*beta))
        numer2 = 0.25*rho*beta*nu*alpha/(F**(1 - beta))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        VolAtm = alpha*(1 + (numer1 + numer2 + numer3)*T)/(F**(1-beta))
        sabrsigma = VolAtm
    else:
        z = (nu/alpha)*((F*X)**(0.5*(1-beta)))*np.log(F/X)
        zhi = np.log((((1 - 2*rho*z + z*z)**0.5) + z - rho)/(1 - rho))
        numer1 = (((1 - beta)**2)/24)*((alpha*alpha)/((F*X)**(1 - beta)))
        numer2 = 0.25*rho*beta*nu*alpha/((F*X)**((1 - beta)/2))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        numer = alpha*(1 + (numer1 + numer2 + numer3)*T)*z
        denom1 = ((1 - beta)**2/24)*(np.log(F/X))**2
        denom2 = (((1 - beta)**4)/1920)*((np.log(F/X))**4)
        denom = ((F*X)**((1 - beta)/2))*(1 + denom1 + denom2)*zhi
        sabrsigma = numer/denom

    return sabrsigma

The definition above contains the function

$$
\begin{split}
{SABR}(F, K, T, \alpha, \beta, \rho, \nu)
\end{split}
$$

How do we determine the parameters $\alpha$, $\rho$ and $\nu$?
- We choose them so that the output of the SABR model matches the implied volatilities observed in the market.
- We refer to this process as "model calibration".

We want to minimize the sum of squared error terms as follows:
  
  $$
    \begin{split}
      \min_{\substack{\alpha,\; \rho,\; \nu}} \;\sum_{i=1}^n \epsilon_i^2
    \end{split}
  $$

We use the "least_squares" algorithm in "scipy" package to calibrate the SABR model parameters:


A suggested optimisation routine is as follows
1. Fix the $\beta$ value (in this assignment: 0.75)
2. At the ATM values, optimise for alpha
3. Finally, at the shifts, optimise for $\rho$ and $\nu$

The SABR model could be fitted using the library pysabr

In [23]:
df_bps

,-0.0200,-0.0150,-0.0100,-0.0050,-0.0025,0.0000,0.0025,0.0050,0.0100,0.0150,0.0200
0,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


In [24]:
# Example Market Data (Strikes and corresponding Implied Volatilities)

market_strikes = np.array([0.02, 0.025, 0.03]) 
market_vols = np.array([0.45, 0.40, 0.38]) / 100 # convert to decimal

# Market data inputs
forward = 0.025271 # current forward rate/price
shift = 0.03       # shift value for shifted SABR (handles negative rates/prices)
t = 10             # time to maturity (years)
beta = 0.5         # fixed beta parameter

# Initialize the SABR object with fixed beta and shift
sabr = pysabr.Hagan2002LognormalSABR(f=forward, shift=shift, t=t, beta=beta)

# Calibrate alpha, rho, and volvol (nu) from the market data
# The calibrate method finds the parameters that best fit the provided strikes and vols
alpha, rho, volvol = sabr.fit(market_strikes, market_vols)

print(f"Calibrated parameters: alpha={alpha:.4f}, rho={rho:.4f}, volvol={volvol:.4f}")

Calibrated parameters: alpha=0.0001, rho=0.5590, volvol=0.0007


In [25]:
def calibrate_sabr(strikes_shift, vols, forward, tenor, beta, shift=0.0):

    strikes = np.asarray(strikes_shift+forward)
    vols = np.asarray(vols)
    # print(strikes)
    # print(vols)

    sabr = pysabr.Hagan2002LognormalSABR(f = forward, shift = shift, t = tenor, beta = beta, v_atm_n = vols[5])
    alpha, rho, nu = sabr.fit(strikes, vols)

    return alpha, rho, nu

In [ ]:
def output_sabr_values(df_bps, forward_series, tenor_series, beta, shift=0.0):
    
    strikes_shift = np.asarray(df_bps.columns)
    
    output_rows = []

    for i in df_bps.index:
        # strikes = strikes_shift + forward_series[i]
        vols = np.asarray(df_bps.iloc[i])
        
        alpha, rho, nu = calibrate_sabr(strikes_shift, vols, forward_series[i], tenor_series[i], beta, shift)

        output_rows.append({
            'alpha': alpha,
            'rho': rho,
            'nu': nu,
            'Tenor': tenor_series[i],
        })
    
    return pd.DataFrame(
                        output_rows,
                        index = df_bps.index,
                        columns = ['alpha', 'rho', 'nu']
                        )


In [42]:
sabr_output_values = output_sabr_values(df_bps, df_forward_rates, df_timings['Tenor'], beta = 0.75)
sabr_output_values

C:\Users\TanFamily4\AppData\Roaming\Python\Python313\site-packages\pysabr\models\hagan_2002_lognormal_sabr.py:82: RuntimeWarning: invalid value encountered in log
  return np.log(a / b)


,alpha,rho,nu
0,0.001588,0.999898,0.000100
1,0.001214,-0.499492,0.018643
2,0.001278,-0.451605,0.015668
3,0.001137,-0.377645,0.011376
4,0.001237,0.999750,0.000100
5,0.001512,0.999899,0.000100
6,0.001527,0.999899,0.000100
7,0.001497,0.999778,0.000100
8,0.001294,0.999849,0.000100
9,0.001171,0.999898,0.000100


## Pricing of swaptions using calibrated SABR and displaced diffusion models

Requirements:
1. 1 x 1 implied vol across K from 1% - 10%
2. 10 x 10 implied vol across K from 1% - 10%
3. 30 x 30 implied vol across K from 1% to 10%

In [ ]:
df_timings

,Expiry,Tenor
0,1.0,1.0
1,1.0,2.0
2,1.0,3.0
3,1.0,5.0
4,1.0,10.0
5,5.0,1.0
6,5.0,2.0
7,5.0,3.0
8,5.0,5.0
9,5.0,10.0


In [39]:
df_bps

,-0.0200,-0.0150,-0.0100,-0.0050,-0.0025,0.0000,0.0025,0.0050,0.0100,0.0150,0.0200
0,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


### Displaced Diffusion

In [41]:
calibration_results_batch

,beta,sigma,residual,success,message,nfev,expiry,forward
instrument,,,,,,,,
0,0.022186,0.002705,0.000716,True,`gtol` termination condition is satisfied.,29,1.0,0.033724
1,0.010219,0.001603,0.000691,True,`gtol` termination condition is satisfied.,47,1.0,0.034525
2,0.010000,0.001614,0.000688,True,`gtol` termination condition is satisfied.,57,1.0,0.035247
3,0.010454,0.001450,0.000704,True,`gtol` termination condition is satisfied.,55,1.0,0.036608
4,0.012284,0.001585,0.000701,True,`gtol` termination condition is satisfied.,40,1.0,0.039007
5,0.010000,0.001723,0.000613,True,`gtol` termination condition is satisfied.,13,5.0,0.039468
6,0.010000,0.001852,0.000580,True,`gtol` termination condition is satisfied.,11,5.0,0.039911
7,0.010000,0.001835,0.000577,True,`gtol` termination condition is satisfied.,11,5.0,0.040339
8,0.010000,0.001539,0.000566,True,`gtol` termination condition is satisfied.,13,5.0,0.041100


### SABR

In [43]:
sabr_output_values

,alpha,rho,nu
0,0.001588,0.999898,0.000100
1,0.001214,-0.499492,0.018643
2,0.001278,-0.451605,0.015668
3,0.001137,-0.377645,0.011376
4,0.001237,0.999750,0.000100
5,0.001512,0.999899,0.000100
6,0.001527,0.999899,0.000100
7,0.001497,0.999778,0.000100
8,0.001294,0.999849,0.000100
9,0.001171,0.999898,0.000100
